<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L49_Advanced_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L49: Advanced Retrieval - Dense, Hybrid, and Re-Ranking

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 49 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Combine dense (embedding) and sparse (BM25) retrieval (hybrid)
- Implement a simple re-ranker on top of retrieved candidates
- Optimize retrieval for RAG quality

---

## Concept: Advanced Retrieval

**Dense retrieval**: embed query and docs, k-NN. **Sparse**: BM25/keyword. **Hybrid**: combine scores (e.g. weighted sum or reciprocal rank fusion). **Re-ranking**: use a cross-encoder or LLM to score query-doc pairs and reorder top-k. Improves RAG accuracy.

---

In [ ]:
!pip install sentence-transformers rank_bm25 -q
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Dense + BM25 (Hybrid)

---

In [ ]:
documents = ["Paris is the capital of France.", "ML is a subset of AI.", "The Eiffel Tower is in Paris.", "Python is used for ML."]
encoder = SentenceTransformer("all-MiniLM-L6-v2")
dense_emb = encoder.encode(documents)
tokenized = [d.split() for d in documents]
bm25 = BM25Okapi(tokenized)

query = "Where is the Eiffel Tower?"
q_emb = encoder.encode([query])[0]
dense_scores = np.dot(dense_emb, q_emb)
bm25_scores = bm25.get_scores(query.split())

alpha = 0.5
dense_n = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min() + 1e-9)
bm25_n = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-9)
hybrid = alpha * dense_n + (1 - alpha) * bm25_n
top = np.argsort(hybrid)[::-1][:2]
print("Hybrid top-2:", [documents[i] for i in top])

## Part 2: Re-Ranking (Cross-Encoder Style)

---

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
pairs = [[query, documents[i]] for i in top]
scores = reranker.predict(pairs)
reranked = [top[i] for i in np.argsort(scores)[::-1]]
print("After re-rank:", [documents[i] for i in reranked])

## Part 3: Reciprocal Rank Fusion (Concept)

---

In [ ]:
# RRF: score(d) = sum 1/(k + rank(d)) over multiple retrievers; k=60 common.
print("RRF combines rankings from dense, BM25, etc. without normalizing scores.")

## Exercises

1. Tune alpha for hybrid (0.3, 0.5, 0.7) on a small QA set.
2. Retrieve top-20, re-rank to top-5; compare RAG answer quality vs top-5 without re-rank.
3. Implement RRF for two rankers (dense and BM25) and compare to single retriever.

---

## Key Takeaways

1. Hybrid (dense + sparse) often beats either alone.
2. Re-ranking with a cross-encoder improves precision for RAG.
3. RRF is a simple way to fuse multiple retrieval runs.

---

## Next Lesson

**L50: Agent Systems** – LLM agents, ReAct, planning.

---